## Install Libraries

In [1]:
!pip install transformers peft bitsandbytes accelerate trl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requir

## Import Libraries

In [2]:
import torch
import json
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, TaskType
from trl import RewardTrainer, RewardConfig

print("Libraries imported!")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Libraries imported!
GPU: Tesla T4


## Load Tokenizer

In [3]:
sft_model_path = "/kaggle/input/datasets/rlhf0226/sft-model-2"

tokenizer = AutoTokenizer.from_pretrained(sft_model_path)
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded!")

Tokenizer loaded!


## Load Reward Training Data

In [4]:
data_path = "/kaggle/input/datasets/rlhf0226/reward-data-train"

def load_jsonl(file):
    with open(file) as f:
        return [json.loads(line) for line in f]

data = load_jsonl(data_path + "/reward_train.jsonl")
print(f"Loaded {len(data)} samples")
print(f"Columns: {list(data[0].keys())}")
print(f"\nSample:")
print(data[0])

Loaded 5320 samples
Columns: ['prompt', 'chosen', 'rejected']

Sample:
{'prompt': 'During the World War II, where was the Canadian exile government located?', 'chosen': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nDuring the World War II, where was the Canadian exile government located?<|im_end|>\n<|im_start|>assistant\nDuring World War II, Canada did not have an exile government. Canada was an active participant in the conflict on the side of the Allies, and its government operated from within the country. Exile governments were established by countries that were occupied by enemy forces, such as the governments of Poland, Belgium, and the Netherlands, which set up exile governments in London. Canada, however, was not occupied and continued to function from its capital, Ottawa.<|im_end|>\n', 'rejected': "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_star

## Format Dataset

In [5]:
def format_dataset(example):
    return {
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(
    format_dataset,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/5320 [00:00<?, ? examples/s]

In [ ]:
print(dataset.column_names)

In [ ]:
dataset[0]

## Train/Validation Split

In [6]:
# Cell 6 — Split stays the same
split      = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data   = split["test"]

print(f"Train: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")

Train: 4788 samples
Val:   532 samples


## Load Model with QLoRA

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading model... (2-3 mins)")
model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded!")

Loading model... (2-3 mins)


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


## LoRA Configuration

In [8]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)

print("LoRA config ready!")

LoRA config ready!


## Training Arguments 

In [10]:
training_args = RewardConfig(
    output_dir="/kaggle/working/reward_model",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_steps=50,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    fp16=False,
    bf16=True,
    report_to="none",
    max_length=640,
    weight_decay=0.01
)

# Fix to prevent error
training_args.center_rewards_coefficient = None

print("Training arguments set!")

Training arguments set!


## Train

In [ ]:
trainer = RewardTrainer(
    model=model,
    args=training_args,
     processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config
)

print("Starting Reward Model Training...")
print("Watch for rewards/chosen going UP and rewards/rejected going DOWN!\n")

trainer.train()

print("Reward Model Training complete!")

Adding EOS to train dataset:   0%|          | 0/4788 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4788 [00:00<?, ? examples/s]

Filtering train >640 tokens:   0%|          | 0/4788 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/532 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/532 [00:00<?, ? examples/s]

Filtering eval >640 tokens:   0%|          | 0/532 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.


Starting Reward Model Training...
Watch for rewards/chosen going UP and rewards/rejected going DOWN!



Step,Training Loss,Validation Loss,Num Tokens,Min Reward,Mean Reward,Max Reward,Accuracy,Margin
50,0.735589,0.729050,872208.000000,-0.822142,0.241217,1.358209,0.485075,0.005837
100,0.703452,0.684846,1731967.000000,-1.496793,-0.506493,0.588461,0.578358,0.094352
150,0.668300,0.669308,2585282.000000,-1.477787,-0.419368,0.692891,0.587687,0.136897
200,0.653941,0.663506,3440666.000000,-1.341768,-0.266717,0.806276,0.585821,0.150808
250,0.665693,0.659631,4313233.000000,-1.497085,-0.405745,0.689886,0.597015,0.161695


## Custom Evaluation

In [ ]:
print("Evaluating Reward Model...")
print("=" * 60)

model.eval()
correct = 0
total = min(200, len(val_data))

for i in range(total):
    sample = val_data[i]

    chosen_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['chosen']}"
    rejected_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['rejected']}"

    chosen_inputs = tokenizer(
        chosen_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    rejected_inputs = tokenizer(
        rejected_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        chosen_score = model(**chosen_inputs).logits.squeeze().item()
        rejected_score = model(**rejected_inputs).logits.squeeze().item()

    is_correct = chosen_score > rejected_score

    if is_correct:
        correct += 1

    if i < 5:
        print(
            f"{i+1}: "
            f"chosen={chosen_score:.4f} "
            f"rejected={rejected_score:.4f} "
            f"correct={is_correct}"
        )

accuracy = 100 * correct / total

print("\n" + "=" * 60)
print(f"Final Accuracy: {accuracy:.2f}% ({correct}/{total})")

## Save Model

In [ ]:
trainer.save_model("/kaggle/working/reward_model")
tokenizer.save_pretrained("/kaggle/working/reward_model")

print("Reward model saved!")
print("Location: /kaggle/working/reward_model")

## Quick Sanity Test

In [ ]:
def get_reward_score(prompt, response):
    text = f"\n\nHuman: {prompt}\n\nAssistant: {response}"
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        score = model(**inputs).logits[0].item()
    return score

# Test it!
prompt       = "How do I stay healthy?"
good_response = "Exercise daily, eat balanced meals, sleep 7-8 hours!"
bad_response  = "I don't know just try stuff"

good_score = get_reward_score(prompt, good_response)
bad_score  = get_reward_score(prompt, bad_response)

print("Reward Scores:")
print(f"Good response: {good_score:.4f}")
print(f"Bad response:  {bad_score:.4f}")

if good_score > bad_score:
    print("\nReward model working correctly!")
else:
    print("\nReward model needs more training")

## Saving zip

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/reward_model",
    "zip",
    "/kaggle/working/reward_model"
)

print("ZIP created!")

In [ ]:
print(type(model))
print(type(trainer.model))

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
model = trainer.model

In [ ]:
print("Evaluating Reward Model...")
print("=" * 60)

model.eval()
correct = 0
total = min(200, len(val_data))

for i in range(total):
    sample = val_data[i]

    chosen_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['chosen']}"
    rejected_text = f"\n\nHuman: {sample['prompt']}\n\nAssistant: {sample['rejected']}"

    chosen_inputs = tokenizer(
        chosen_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    rejected_inputs = tokenizer(
        rejected_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        chosen_score = model(**chosen_inputs).logits[0].item()
        rejected_score = model(**rejected_inputs).logits[0].item()

    is_correct = chosen_score > rejected_score

    if is_correct:
        correct += 1

    if i < 5 or i >= total - 3:
        status = "✅" if is_correct else "❌"
        print(
            f"Row {i+1:3d} | "
            f"chosen: {chosen_score:7.3f} | "
            f"rejected: {rejected_score:7.3f} | "
            f"correct: {str(is_correct):<5} {status}"
        )
    elif i == 5:
        print("...")

accuracy = correct / total * 100

print("=" * 60)
print(f"\nFinal Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")